In [ ]:
import numpy as np
import pandas as pd
from pgmpy.base import DAG
from pgmpy.estimators import PC
from pgmpy.prediction import NaiveAdjustmentRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# --- Step 1: Generate Raw Continuous Data ---
np.random.seed(42)
num_samples = 2500

# Continuous graph structure variables
Conf_Z = np.random.normal(50, 10, size=num_samples)
Feat_X = 1.8 * Conf_Z + np.random.normal(0, 2, size=num_samples)

# Continuous target variable ('Class' continuous regressor node)
Target_Continuous = 2.5 * Feat_X - 1.2 * Conf_Z + np.random.normal(0, 1, size=num_samples)

# Downstream Effect variable (Child of Target)
Child_Y = 3.0 * Target_Continuous + np.random.normal(0, 2, size=num_samples)
Noise = np.random.normal(100, 20, size=num_samples)

df = pd.DataFrame({
    "Feat_X": Feat_X, "Conf_Z": Conf_Z, 
    "Target": Target_Continuous, "Child_Y": Child_Y, "Noise": Noise
})

# --- Step 2: Discover Causal Graph Structure ---
est = PC(data=df)
learned_dag = est.estimate(ci_test="pearsonr", return_type="dag")

# --- Step 3: Isolate features within the Markov Blanket ---
mb_features = list(learned_dag.get_markov_blanket("Target"))
print(f"Features mapped inside Markov Blanket: {mb_features}")

# --- Step 4: Configure the Native Causal Regressor Graph ---
# We define a sub-graph containing only the Target and its structural Blanket
prediction_subgraph = learned_dag.subgraph(mb_features + ["Target"])

# Assign explicit structural roles for the adjustment layer
# Exposures = variables directly driving the outcome (Parents)
# Adjustment = variables that act as spouses/confounders within the sub-graph
parents = list(learned_dag.get_parents("Target"))
spouses_and_children = [node for node in mb_features if node not in parents]

causal_graph_model = DAG(
    prediction_subgraph.edges(),
    roles={
        "exposures": parents,
        "outcomes": ["Target"],
        "adjustment": spouses_and_children
    }
)

# --- Step 5: Split Data and Fit the Regressor ---
X = df[mb_features]
y = df["Target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the Native pgmpy scikit-learn compatible Regressor
# By default, it uses LinearRegression as the base estimator
regressor = NaiveAdjustmentRegressor(causal_graph=causal_graph_model)

# Train directly using the layout defined by the DAG
regressor.fit(X_train, y_train)

# --- Step 6: Predict and Evaluate Performance ---
predictions = regressor.predict(X_test)

print("\n--- Structural Regressor Metrics ---")
print(f"Model R² Score: {r2_score(y_test, predictions):.4f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, predictions):.4f}")
